### Приоритизация обращений с использованием HistGradientBoosting и расширенных признаков из событийной истории пользователей

#### Цель: 
   Предсказать вероятность приоритетности обращения (target=1) для каждого lead_id
   на основе его событийной истории до момента назначения (assignment_ts).

#### Описание подхода:
   - Используются только события с event_ts < assignment_ts для предотвращения утечки данных.
   - Строятся расширенные признаки, отражающие вовлеченность пользователя:
     * давность и длительность активности,
     * разнообразие действий (типы событий, контексты),
     * ускорение активности (сравнение интенсивности в последние 3 дня vs 3-7 дней назад),
     * средние интервалы между событиями,
     * статистика по просмотренным ценам и позициям в выдаче,
     * счетчики по типам событий.
   - Модель: HistGradientBoostingClassifier с поддержкой категориальных признаков, ранней остановкой и балансировкой классов.

#### Валидация:
   - Временной сплит по assignment_date (последние 20% дат),
   - Метрика: Daily Average Precision (усреднение AP по дням).

#### Результаты:
   - Daily AP = 0.63173, Average Precision = 0.63061 (рез-т со Stepik)
   - Топ-признаки: разнообразие контекстов (n_distinct_ctx_seq), последний контекст/событие, разброс цен просмотренных объявлений.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from sklearn.inspection import permutation_importance

ROOT = Path('.')
DATA_DIR = ROOT / 'data'
SUBMISSIONS_DIR = ROOT / 'submissions'

RANDOM_STATE = 42
ID_COLUMNS = {"lead_id", "user_id"}
TIME_COLUMNS = {"assignment_ts", "assignment_date"}
TARGET = "target"

In [43]:
def load_data():
    train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["assignment_ts", "assignment_date"])
    test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["assignment_ts", "assignment_date"])
    events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_ts"])
    return train, test, events

train, test, events = load_data()
print("train:", train.shape, "test:", test.shape)

train: (13694, 119) test: (4306, 118)


In [44]:
def build_event_features(leads_df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """
    строит дополнительные признаки из событийной истории с предотвращением утечки:
    используются только события с event_ts < assignment_ts
    """
    leads = leads_df[["lead_id", "assignment_ts"]].copy()
    ev = events.merge(leads, on="lead_id", how="inner")
    ev = ev[ev["event_ts"] < ev["assignment_ts"]].copy()  # против утечки
    
    ev["hours_before_assignment"] = (
        ev["assignment_ts"] - ev["event_ts"]
    ).dt.total_seconds() / 3600.0
    ev = ev.sort_values(["lead_id", "event_ts"])
    feats = []
    
    # recency = время с последнего/первого события
    g = ev.groupby("lead_id")["hours_before_assignment"]
    recency = g.agg(
        hours_since_last_event="min",
        hours_since_first_event="max",
    )
    recency["event_history_span_hours"] = (
        recency["hours_since_first_event"] - recency["hours_since_last_event"]
    )
    feats.append(recency)
    
    # объем и разнообразие
    vol = ev.groupby("lead_id").size().to_frame("total_events_all_time")
    diversity = ev.groupby("lead_id").agg(
        n_distinct_event_types=("event_type", "nunique"),
        n_distinct_ctx_seq=("ctx_seq", "nunique"),
    )
    feats.append(vol)
    feats.append(diversity)
    
    # последнее действие перед назначением
    last_event = (
        ev.sort_values("event_ts").groupby("lead_id").tail(1)
        .set_index("lead_id")[["event_type", "ctx_seq"]]
        .rename(columns={"event_type": "last_event_type", "ctx_seq": "last_ctx_seq"})
    )
    feats.append(last_event)
    
    # ускорение активности = события за последние 3 дня vs предыдущие 3-7 дней
    ev["in_last_3d"] = ev["hours_before_assignment"] <= 72
    ev["in_prior_3_7d"] = (ev["hours_before_assignment"] > 72) & (ev["hours_before_assignment"] <= 168)
    accel = ev.groupby("lead_id").agg(
        events_last_3d=("in_last_3d", "sum"),
        events_prior_3_7d=("in_prior_3_7d", "sum"),
    )
    accel["activity_acceleration"] = (accel["events_last_3d"] + 1) / (accel["events_prior_3_7d"] + 1)
    feats.append(accel[["activity_acceleration"]])
    
    # межсобытийные интервалы как прокси плотности вовлеченности
    def gap_stats(group):
        ts = group.sort_values("event_ts")["event_ts"]
        if len(ts) < 2:
            return pd.Series({"mean_gap_hours": np.nan, "min_gap_hours": np.nan})
        gaps = ts.diff().dropna().dt.total_seconds() / 3600.0
        return pd.Series({"mean_gap_hours": gaps.mean(), "min_gap_hours": gaps.min()})
    gaps = ev.groupby("lead_id").apply(gap_stats, include_groups=False)
    feats.append(gaps)
    
    # разброс цен просмотренных объявлений
    price = ev.groupby("lead_id")["item_price_log"].agg(
        browsed_price_mean="mean",
        browsed_price_std="std",
    )
    feats.append(price)
    
    # позиция в выдаче (src_slot) как прокси вовлеченности
    slot = ev.groupby("lead_id")["src_slot"].agg(
        avg_src_slot="mean",
        min_src_slot="min",
    )
    feats.append(slot)
    
    # счетчики по типам событий (сверка с готовыми агрегатами)
    type_counts = (
        ev.groupby(["lead_id", "event_type"]).size().unstack(fill_value=0)
    )
    type_counts.columns = [f"evt_count_{c}" for c in type_counts.columns]
    feats.append(type_counts)
    
    out = pd.concat(feats, axis=1).reset_index()
    out = out.rename(columns={"index": "lead_id"})
    return out

def add_event_features(df: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    ev_feats = build_event_features(df, events)
    merged = df.merge(ev_feats, on="lead_id", how="left")
    return merged

In [45]:
train = add_event_features(train, events)
test = add_event_features(test, events)
print("[after event features] train:", train.shape, "test:", test.shape)

[after event features] train: (13694, 139) test: (4306, 138)


In [46]:
# подготовка признаков для обучения модели

def get_feature_columns(train_df: pd.DataFrame, test_df: pd.DataFrame):
    non_feature = ID_COLUMNS | TIME_COLUMNS | {TARGET, "split"}
    feature_cols = [
        c for c in train_df.columns
        if c not in non_feature and c in test_df.columns
    ]
    categorical_cols = [
        c for c in feature_cols
        if not pd.api.types.is_numeric_dtype(train_df[c])
    ]
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]
    return feature_cols, numeric_cols, categorical_cols

def prep_categoricals(df: pd.DataFrame, categorical_cols):
    df = df.copy()
    for c in categorical_cols:
        df[c] = df[c].astype("category")
    return df

feature_cols, numeric_cols, categorical_cols = get_feature_columns(train, test)
print(f"features: {len(feature_cols)} (numeric={len(numeric_cols)}, categorical={len(categorical_cols)})")

train = prep_categoricals(train, categorical_cols)
test = prep_categoricals(test, categorical_cols)

# выравнивание категорий между train и test
for c in categorical_cols:
    cats = pd.api.types.union_categoricals([train[c], test[c]]).categories
    train[c] = train[c].cat.set_categories(cats)
    test[c] = test[c].cat.set_categories(cats)

features: 134 (numeric=125, categorical=9)


In [47]:
def daily_average_precision(df: pd.DataFrame, y_true_col: str, score_col: str, date_col: str) -> float:
    """считает average precision отдельно по каждой дате, усредняет по дням"""
    aps = []
    for _, day_df in df.groupby(date_col):
        y = day_df[y_true_col].values
        s = day_df[score_col].values
        if y.sum() == 0:
            continue
        ap = average_precision_score(y, s)
        aps.append(ap)
    return float(np.mean(aps))

In [ ]:
# временной валидационный сплит
def make_time_split(train_df: pd.DataFrame, valid_frac=0.2):
    dates = train_df["assignment_date"].dt.date
    ordered_dates = sorted(dates.unique())
    cutoff = ordered_dates[int(len(ordered_dates) * (1 - valid_frac))]
    train_part = train_df[dates < cutoff]
    valid_part = train_df[dates >= cutoff]
    return train_part, valid_part

train_part, valid_part = make_time_split(train, valid_frac=0.2)
print("train_part:", train_part.shape, "valid_part:", valid_part.shape)

train_part: (10272, 139) valid_part: (3422, 139)


In [49]:
# обучение

def build_model(categorical_cols, feature_cols):
    cat_indices = [feature_cols.index(c) for c in categorical_cols]
    model = HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        categorical_features=cat_indices,
        max_iter=600,
        learning_rate=0.03,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        class_weight="balanced",
    )
    return model

model = build_model(categorical_cols, feature_cols)
model.fit(train_part[feature_cols], train_part[TARGET])

# оценка на валидации
valid_scores = model.predict_proba(valid_part[feature_cols])[:, 1]
valid_ap_overall = average_precision_score(valid_part[TARGET], valid_scores)

valid_eval = valid_part[["assignment_date", TARGET]].copy()
valid_eval["score"] = valid_scores
valid_daily_ap = daily_average_precision(valid_eval, TARGET, "score", "assignment_date")

print(f"\nvalidation overall AP:  {valid_ap_overall:.5f}")
print(f"validation Daily AP:    {valid_daily_ap:.5f}")


validation overall AP:  0.64853
validation Daily AP:    0.65136


In [50]:
# важность признаков

perm = permutation_importance(
    model, valid_part[feature_cols], valid_part[TARGET],
    n_repeats=3, random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
}).sort_values("importance_mean", ascending=False)

print("\nтоп-20 признаков по важности (убывание AP при перемешивании):")
print(importance_df.head(20).to_string(index=False))
importance_df.to_csv(MODELS_DIR / "feature_importance.csv", index=False)


топ-20 признаков по важности (убывание AP при перемешивании):
                feature  importance_mean
     n_distinct_ctx_seq         0.144671
           last_ctx_seq         0.066193
        last_event_type         0.033545
      browsed_price_std         0.033483
            lead_source         0.021758
   seller_page_views_7d         0.019450
  seller_page_views_14d         0.015489
       search_views_90d         0.013212
  seller_page_views_30d         0.012650
       photo_swipes_90d         0.008611
 hours_since_last_event         0.007429
           min_src_slot         0.007252
     detail_expands_90d         0.007159
  activity_acceleration         0.006322
         mean_gap_hours         0.006181
  query_refinements_90d         0.005295
similar_item_clicks_90d         0.004305
            call_center         0.003760
  total_events_all_time         0.003496
      user_contacts_90d         0.003300


In [52]:
# финальное обучение на всем train и предсказание на test
# + сохранение файлов

final_model = build_model(categorical_cols, feature_cols)
final_model.fit(train[feature_cols], train[TARGET])

test_scores = final_model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({
    "lead_id": test["lead_id"].astype(str),
    "score": test_scores,
})

# минимал проверки соответствия формату
sample_sub = pd.read_csv(SUBMISSIONS_DIR / "submission.csv")
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(test)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()
assert set(submission["lead_id"]) == set(sample_sub["lead_id"])

submission.to_csv(SUBMISSIONS_DIR / "submission.csv", index=False)
print("\nsubmission.csv записан:", submission.shape)
print(submission.head())


submission.csv записан: (4306, 2)
                 lead_id     score
0  lead_97e409eb8f8c8246  0.174298
1  lead_55310edb4489f9e9  0.468508
2  lead_e7f653a2c6a7eee8  0.559864
3  lead_22f8e1cfc487ac20  0.094960
4  lead_48b638b839abfac3  0.321107


In [53]:
print(f"файл сохранен: {SUBMISSIONS_DIR / 'submission.csv'}")

файл сохранен: C:\Users\user\Documents\data_science_bootcamp_2026_1_gy\submissions\submission.csv
